In [1]:
# 파라미터 테스트용 Lightgbm 및 피쳐 중요도
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. 전처리 함수 (안정적인 V2 상태 유지, Test 배제)
# ==========================================
def preprocess_fast(train_path, layout_path):
    train = pd.read_csv(train_path)
    layout = pd.read_csv(layout_path)
    
    train = pd.merge(train, layout, on='layout_id', how='left')
    train.drop('ID', axis=1, inplace=True)

    target_col = 'avg_delay_minutes_next_30m'
    y_train = train[target_col]
    X_train = train.drop(target_col, axis=1)
    
    # 결측치 처리 (중앙값)
    num_cols = X_train.select_dtypes(include=['float64', 'int64']).columns
    for col in num_cols:
        X_train[col].fillna(X_train[col].median(), inplace=True)

    # 원핫 인코딩
    X_train = pd.get_dummies(X_train, columns=['layout_type'], drop_first=False)

    df = X_train
    
    # 기본 V2 파생 변수
    df['shift_hour_sin'] = np.sin(2 * np.pi * df['shift_hour'] / 24)
    df['shift_hour_cos'] = np.cos(2 * np.pi * df['shift_hour'] / 24)
    df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
    
    if 'robot_utilization' in df.columns and 'congestion_score' in df.columns:
        df['robot_util_x_congestion'] = df['robot_utilization'] * df['congestion_score']
    if 'order_inflow_15m' in df.columns and 'robot_active' in df.columns:
        df['order_pressure'] = df['order_inflow_15m'] / (df['robot_active'] + 1)
    if 'low_battery_ratio' in df.columns and 'charge_queue_length' in df.columns:
        df['battery_stress'] = df['low_battery_ratio'] * df['charge_queue_length']
    if 'congestion_score' in df.columns and 'blocked_path_15m' in df.columns:
        df['congestion_impact'] = df['congestion_score'] * df['blocked_path_15m']

    return X_train, y_train

# ==========================================
# 2. LightGBM 단일 모델 검증 및 피처 중요도 추출
# ==========================================
def validate_lgbm_fast(X_train, y_train):
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    oof_predictions = np.zeros(len(X_train))
    
    # 변수 중요도를 누적할 딕셔너리
    feature_importance_dict = {}
    
    # 이전에 찾은 최적 파라미터 적용
    lgb_params = {
        'objective': 'mae',
        'metric': 'mae',
        'n_estimators': 1500, 
        'learning_rate': 0.04165780165479688,
        'max_depth': 10,
        'num_leaves': 84,
        'subsample': 0.9282065869845333,
        'colsample_bytree': 0.7071982532616117,
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }
    
    SMOOTH_WEIGHT = 10 
    
    print("=== LightGBM 단일 모델 빠른 검증 시작 ===")
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
        x_tr = X_train.iloc[train_idx].copy()
        y_tr = y_train.iloc[train_idx]
        x_val = X_train.iloc[val_idx].copy()
        y_val = y_train.iloc[val_idx]
        
        overall_mean = y_tr.mean()
        
        # Smoothed Target Encoding
        for col in ['scenario_id', 'layout_id']:
            group_stats = y_tr.groupby(x_tr[col]).agg(mean=('mean'), count=('count'))
            smoothed_means = (group_stats['mean'] * group_stats['count'] + overall_mean * SMOOTH_WEIGHT) / (group_stats['count'] + SMOOTH_WEIGHT)
            
            x_tr[f'{col}_target_enc'] = x_tr[col].map(smoothed_means)
            x_val[f'{col}_target_enc'] = x_val[col].map(smoothed_means).fillna(overall_mean)
            
            x_tr.drop(col, axis=1, inplace=True)
            x_val.drop(col, axis=1, inplace=True)
            
        # LightGBM 학습
        model = lgb.LGBMRegressor(**lgb_params)
        model.fit(
            x_tr, y_tr,
            eval_set=[(x_val, y_val)],
            callbacks=[lgb.early_stopping(50, verbose=False)]
        )
        
        # 검증 세트 예측
        val_preds = model.predict(x_val)
        oof_predictions[val_idx] = val_preds
        
        # 중요도 누적
        for idx, feature in enumerate(x_tr.columns):
            if feature not in feature_importance_dict:
                feature_importance_dict[feature] = 0.0
            feature_importance_dict[feature] += model.feature_importances_[idx] / kf.n_splits
            
        print(f"Fold {fold+1} MAE: {mean_absolute_error(y_val, val_preds):.4f}")
        
    total_mae = mean_absolute_error(y_train, oof_predictions)
    
    print("\n" + "="*50)
    print(f"🚀 LightGBM 최종 OOF MAE: {total_mae:.4f}")
    print("="*50 + "\n")
    
    # 중요도 결과 정렬 및 출력
    fi_df = pd.DataFrame({
        'feature': list(feature_importance_dict.keys()),
        'importance': list(feature_importance_dict.values())
    }).sort_values(by='importance', ascending=False)
    
    print("=== 상위 20개 피처 중요도 (Feature Importance) ===")
    print(fi_df.head(20).to_string(index=False))

# ==========================================
# 3. 메인 실행부
# ==========================================
if __name__ == "__main__":
    TRAIN_PATH = 'train.csv' 
    LAYOUT_PATH = 'layout_info.csv'
    
    X_train, y_train = preprocess_fast(TRAIN_PATH, LAYOUT_PATH)
    validate_lgbm_fast(X_train, y_train)

=== LightGBM 단일 모델 빠른 검증 시작 ===
Fold 1 MAE: 4.7400
Fold 2 MAE: 4.8571
Fold 3 MAE: 4.7089
Fold 4 MAE: 4.7739
Fold 5 MAE: 4.7334

🚀 LightGBM 최종 OOF MAE: 4.7627

=== 상위 20개 피처 중요도 (Feature Importance) ===
                feature  importance
 scenario_id_target_enc      3224.8
           battery_mean      1902.2
       pack_utilization      1839.4
      avg_trip_distance      1675.4
       order_inflow_15m      1666.4
         order_pressure      1631.6
            battery_std      1499.0
    avg_items_per_order      1483.8
      sku_concentration      1482.6
outbound_truck_wait_min      1426.8
   pick_list_length_avg      1419.0
path_optimization_score      1399.2
         unique_sku_15m      1394.8
   fleet_age_months_avg      1377.4
       bulk_order_ratio      1370.4
         wifi_signal_db      1364.4
      staging_area_util      1362.6
        backorder_ratio      1357.6
      loading_dock_util      1353.6
daily_forecast_accuracy      1346.6


In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

def run_everything(train_path, layout_path):
    print("1. 데이터 로드 및 전처리 시작...")
    train = pd.read_csv(train_path)
    layout = pd.read_csv(layout_path)
    
    train = pd.merge(train, layout, on='layout_id', how='left')
    train.drop('ID', axis=1, inplace=True)

    target_col = 'avg_delay_minutes_next_30m'
    y_train = train[target_col]
    X_train = train.drop(target_col, axis=1)
    
    # 결측치 처리
    for col in X_train.select_dtypes(include=['float64', 'int64']).columns:
        X_train[col].fillna(X_train[col].median(), inplace=True)

    # 파생 변수: 주기성 부여 및 원본 삭제 (노이즈 차단!)
    X_train['shift_hour_sin'] = np.sin(2 * np.pi * X_train['shift_hour'] / 24)
    X_train['shift_hour_cos'] = np.cos(2 * np.pi * X_train['shift_hour'] / 24)
    X_train['day_of_week_sin'] = np.sin(2 * np.pi * X_train['day_of_week'] / 7)
    X_train['day_of_week_cos'] = np.cos(2 * np.pi * X_train['day_of_week'] / 7)
    X_train.drop(['shift_hour', 'day_of_week'], axis=1, inplace=True) 
    
    if 'robot_utilization' in X_train.columns and 'congestion_score' in X_train.columns:
        X_train['robot_util_x_congestion'] = X_train['robot_utilization'] * X_train['congestion_score']
        
    X_train = pd.get_dummies(X_train, columns=['layout_type'], drop_first=False)

    print("2. 창고 상태(쾌적/혼잡/마비) 3분할 군집화 진행 중...")
    # 군집화 핵심 변수 (안전장치 추가)
    possible_features = ['robot_utilization', 'congestion_score', 'charge_queue_length', 'task_reassign_15m']
    cluster_features = [f for f in possible_features if f in X_train.columns]
    
    X_cluster = X_train[cluster_features].copy()
    X_cluster.replace([np.inf, -np.inf], np.nan, inplace=True)
    X_cluster.fillna(0, inplace=True)
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_cluster)
    
    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    X_train['cluster_id'] = kmeans.fit_predict(X_scaled)

    # ----------------------------------------------------
    print("3. 군집별 분할 정복(Divide & Conquer) 모델 학습 시작...")
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    oof_predictions = np.zeros(len(X_train))
    
    lgb_params = {
        'objective': 'mae', 'metric': 'mae', 'n_estimators': 1500, 
        'learning_rate': 0.0416, 'max_depth': 10, 'num_leaves': 84,
        'subsample': 0.928, 'colsample_bytree': 0.707,
        'random_state': 42, 'n_jobs': -1, 'verbose': -1
    }
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
        x_tr = X_train.iloc[train_idx].copy()
        y_tr = y_train.iloc[train_idx]
        x_val = X_train.iloc[val_idx].copy()
        y_val = y_train.iloc[val_idx]
        
        overall_mean = y_tr.mean()
        
        # Target Encoding
        for col in ['scenario_id', 'layout_id']:
            if col in x_tr.columns:
                group_stats = y_tr.groupby(x_tr[col]).agg(mean=('mean'), count=('count'))
                smoothed_means = (group_stats['mean'] * group_stats['count'] + overall_mean * 10) / (group_stats['count'] + 10)
                
                x_tr[f'{col}_target_enc'] = x_tr[col].map(smoothed_means)
                x_val[f'{col}_target_enc'] = x_val[col].map(smoothed_means).fillna(overall_mean)
                
                x_tr.drop(col, axis=1, inplace=True)
                x_val.drop(col, axis=1, inplace=True)

        fold_preds = np.zeros(len(x_val))
        
        # 군집별로 3개의 LightGBM 모델 따로 학습
        for cid in range(3):
            tr_mask = x_tr['cluster_id'] == cid
            val_mask = x_val['cluster_id'] == cid
            
            if not val_mask.any() or not tr_mask.any(): 
                continue
            
            model = lgb.LGBMRegressor(**lgb_params)
            model.fit(
                x_tr[tr_mask].drop('cluster_id', axis=1), y_tr[tr_mask],
                eval_set=[(x_val[val_mask].drop('cluster_id', axis=1), y_val[val_mask])],
                callbacks=[lgb.early_stopping(50, verbose=False)]
            )
            
            fold_preds[val_mask] = model.predict(x_val[val_mask].drop('cluster_id', axis=1))
            
        oof_predictions[val_idx] = fold_preds
        print(f"  - Fold {fold+1} MAE: {mean_absolute_error(y_val, fold_preds):.4f}")
        
    total_mae = mean_absolute_error(y_train, oof_predictions)
    print("\n" + "="*50)
    print(f"🚀 [분할정복] 최종 OOF MAE: {total_mae:.4f}")
    print("="*50 + "\n")

# 여기서 파일명만 로컬 환경에 맞게 적어주세요!
if __name__ == "__main__":
    TRAIN_PATH = 'sampled_train.csv' 
    LAYOUT_PATH = 'layout_info.csv'
    run_everything(TRAIN_PATH, LAYOUT_PATH)

1. 데이터 로드 및 전처리 시작...
2. 창고 상태(쾌적/혼잡/마비) 3분할 군집화 진행 중...
3. 군집별 분할 정복(Divide & Conquer) 모델 학습 시작...
  - Fold 1 MAE: 4.7031
  - Fold 2 MAE: 4.7982
  - Fold 3 MAE: 4.6859
  - Fold 4 MAE: 4.7483
  - Fold 5 MAE: 4.7267

🚀 [분할정복] 최종 OOF MAE: 4.7324



In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# optuna 로그 숨기기 (너무 길어지는 것 방지)
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ==========================================
# 1. 완벽 전처리 및 군집화 (Tree 전용)
# ==========================================
def preprocess_for_tree(train_path, layout_path):
    print("\n1. 데이터 로드 및 4대 핵심 파생 변수 전처리...")
    train = pd.read_csv(train_path)
    layout = pd.read_csv(layout_path)
    
    train = pd.merge(train, layout, on='layout_id', how='left')
    train.drop('ID', axis=1, inplace=True)

    target_col = 'avg_delay_minutes_next_30m'
    y_train = train[target_col].copy()
    X_train = train.drop(target_col, axis=1)
    
    # 결측치 처리
    for col in X_train.select_dtypes(include=['float64', 'int64']).columns:
        X_train[col].fillna(X_train[col].median(), inplace=True)

    # 1. 주기성 부여 및 원본 삭제
    if 'shift_hour' in X_train.columns:
        X_train['shift_hour_sin'] = np.sin(2 * np.pi * X_train['shift_hour'] / 24)
        X_train['shift_hour_cos'] = np.cos(2 * np.pi * X_train['shift_hour'] / 24)
    if 'day_of_week' in X_train.columns:
        X_train['day_of_week_sin'] = np.sin(2 * np.pi * X_train['day_of_week'] / 7)
        X_train['day_of_week_cos'] = np.cos(2 * np.pi * X_train['day_of_week'] / 7)
    X_train.drop(['shift_hour', 'day_of_week'], axis=1, inplace=True, errors='ignore') 
    
    # 2. 4대 핵심 파생 변수 추가
    if 'robot_utilization' in X_train.columns and 'congestion_score' in X_train.columns:
        X_train['robot_util_x_congestion'] = X_train['robot_utilization'] * X_train['congestion_score']
    if 'order_inflow_15m' in X_train.columns and 'robot_active' in X_train.columns:
        X_train['order_pressure'] = X_train['order_inflow_15m'] / (X_train['robot_active'] + 1)
    if 'low_battery_ratio' in X_train.columns and 'charge_queue_length' in X_train.columns:
        X_train['battery_stress'] = X_train['low_battery_ratio'] * X_train['charge_queue_length']
    if 'congestion_score' in X_train.columns and 'blocked_path_15m' in X_train.columns:
        X_train['congestion_impact'] = X_train['congestion_score'] * X_train['blocked_path_15m']
        
    X_train = pd.get_dummies(X_train, columns=['layout_type'], drop_first=False)

    print("2. 3분할(쾌적/혼잡/마비) K-Means 군집화...")
    cluster_features = [f for f in ['robot_utilization', 'congestion_score', 'charge_queue_length', 'task_reassign_15m'] if f in X_train.columns]
    
    X_cluster = X_train[cluster_features].copy()
    X_cluster.replace([np.inf, -np.inf], np.nan, inplace=True)
    X_cluster.fillna(0, inplace=True)
    
    scaler_cluster = StandardScaler()
    X_scaled_cluster = scaler_cluster.fit_transform(X_cluster)
    
    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    X_train['cluster_id'] = kmeans.fit_predict(X_scaled_cluster)
    
    X_train.replace([np.inf, -np.inf], np.nan, inplace=True)
    X_train.fillna(0, inplace=True)
    
    return X_train, y_train

# ==========================================
# 2. 군집별 Optuna 튜닝 및 최종 평가 (수정본)
# ==========================================
def optimize_and_evaluate(X_train, y_train):
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    best_params_per_cluster = {}
    
    print("\n3. 군집별 Optuna 극한 튜닝 시작 (각 군집당 15번 탐색)...")
    
    # ---------------------------------------------------------
    # [Step 1] 각 군집별로 가장 좋은 파라미터 찾기
    # ---------------------------------------------------------
    for cid in range(3):
        mask = X_train['cluster_id'] == cid
        if not mask.any(): continue
            
        x_c = X_train[mask].drop('cluster_id', axis=1)
        y_c = y_train[mask]
        
        def objective(trial):
            params = {
                'objective': 'mae',
                'metric': 'mae',
                'boosting_type': 'gbdt',
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
                'max_depth': trial.suggest_int('max_depth', 4, 12),
                'num_leaves': trial.suggest_int('num_leaves', 15, 127),
                'subsample': trial.suggest_float('subsample', 0.7, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
                'n_estimators': 500, # 빠른 탐색용
                'n_jobs': -1,
                'verbose': -1,
                'random_state': 42
            }
            
            cv_mae = []
            for tr_idx, val_idx in KFold(n_splits=3, shuffle=True, random_state=42).split(x_c):
                x_tr, y_tr = x_c.iloc[tr_idx].copy(), y_c.iloc[tr_idx].copy()
                x_val, y_val = x_c.iloc[val_idx].copy(), y_c.iloc[val_idx].copy()
                
                overall_mean = y_tr.mean()
                
                # [핵심 수정] Optuna 탐색 중에도 문자열 변수를 숫자로 치환! (Target Encoding)
                for col in ['scenario_id', 'layout_id']:
                    if col in x_tr.columns:
                        group_stats = y_tr.groupby(x_tr[col]).agg(mean=('mean'), count=('count'))
                        smoothed_means = (group_stats['mean'] * group_stats['count'] + overall_mean * 10) / (group_stats['count'] + 10)
                        x_tr[col] = x_tr[col].map(smoothed_means)
                        x_val[col] = x_val[col].map(smoothed_means).fillna(overall_mean)
                
                model = lgb.LGBMRegressor(**params)
                model.fit(x_tr, y_tr, eval_set=[(x_val, y_val)], callbacks=[lgb.early_stopping(30, verbose=False)])
                cv_mae.append(mean_absolute_error(y_val, model.predict(x_val)))
                
            return np.mean(cv_mae)

        study = optuna.create_study(direction='minimize')
        study.optimize(objective, n_trials=15)
        best_params_per_cluster[cid] = study.best_params
        best_params_per_cluster[cid]['n_estimators'] = 1500 
        best_params_per_cluster[cid]['objective'] = 'mae'
        best_params_per_cluster[cid]['n_jobs'] = -1
        best_params_per_cluster[cid]['verbose'] = -1
        
        print(f"  > Cluster {cid} 최적 파라미터 찾기 완료 (MAE: {study.best_value:.4f})")

    # ---------------------------------------------------------
    # [Step 2] 찾은 파라미터로 최종 5-Fold 전체 평가
    # ---------------------------------------------------------
    print("\n4. 최적 파라미터 장착 후 최종 5-Fold 검증 시작...")
    oof_predictions = np.zeros(len(X_train))
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
        x_tr, y_tr = X_train.iloc[train_idx].copy(), y_train.iloc[train_idx]
        x_val, y_val = X_train.iloc[val_idx].copy(), y_train.iloc[val_idx]
        
        overall_mean = y_tr.mean()
        
        # Target Encoding
        for col in ['scenario_id', 'layout_id']:
            if col in x_tr.columns:
                group_stats = y_tr.groupby(x_tr[col]).agg(mean=('mean'), count=('count'))
                smoothed_means = (group_stats['mean'] * group_stats['count'] + overall_mean * 10) / (group_stats['count'] + 10)
                # 새로운 컬럼 생성 대신 원본 덮어쓰기로 수정 (코드 간소화 및 에러 방지)
                x_tr[col] = x_tr[col].map(smoothed_means)
                x_val[col] = x_val[col].map(smoothed_means).fillna(overall_mean)

        fold_preds = np.zeros(len(x_val))
        
        for cid in range(3):
            tr_mask = x_tr['cluster_id'] == cid
            val_mask = x_val['cluster_id'] == cid
            if not val_mask.any() or not tr_mask.any(): continue
            
            model = lgb.LGBMRegressor(**best_params_per_cluster[cid])
            model.fit(
                x_tr[tr_mask].drop('cluster_id', axis=1), y_tr[tr_mask],
                eval_set=[(x_val[val_mask].drop('cluster_id', axis=1), y_val[val_mask])],
                callbacks=[lgb.early_stopping(50, verbose=False)]
            )
            fold_preds[val_mask] = model.predict(x_val[val_mask].drop('cluster_id', axis=1))
            
        oof_predictions[val_idx] = fold_preds
        print(f"  - Fold {fold+1} 완료 (MAE: {mean_absolute_error(y_val, fold_preds):.4f})")
        
    total_mae = mean_absolute_error(y_train, oof_predictions)
    print("\n" + "="*50)
    print(f"🏆 [순정 LGBM + 극한 튜닝] 최종 OOF MAE: {total_mae:.4f}")
    print("="*50 + "\n")

if __name__ == "__main__":
    TRAIN_PATH = 'train.csv' # 실제 파일명 확인
    LAYOUT_PATH = 'layout_info.csv'
    
    X_train, y_train = preprocess_for_tree(TRAIN_PATH, LAYOUT_PATH)
    optimize_and_evaluate(X_train, y_train)

c:\Users\kyw41\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



1. 데이터 로드 및 4대 핵심 파생 변수 전처리...
2. 3분할(쾌적/혼잡/마비) K-Means 군집화...

3. 군집별 Optuna 극한 튜닝 시작 (각 군집당 15번 탐색)...
  > Cluster 0 최적 파라미터 찾기 완료 (MAE: 3.9970)
  > Cluster 1 최적 파라미터 찾기 완료 (MAE: 8.6345)
  > Cluster 2 최적 파라미터 찾기 완료 (MAE: 4.2096)

4. 최적 파라미터 장착 후 최종 5-Fold 검증 시작...
  - Fold 1 완료 (MAE: 4.6988)
  - Fold 2 완료 (MAE: 4.7914)
  - Fold 3 완료 (MAE: 4.6787)
  - Fold 4 완료 (MAE: 4.7305)
  - Fold 5 완료 (MAE: 4.6988)

🏆 [순정 LGBM + 극한 튜닝] 최종 OOF MAE: 4.7196



In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.cluster import KMeans
from sklearn.preprocessing import RobustScaler
import warnings
import gc

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ==========================================
# [수정됨] 메모리 사용량 절반으로 줄이는 함수 (문자열 충돌 방지)
# ==========================================
def reduce_mem_usage(df):
    for col in df.columns:
        col_type = df[col].dtype
        
        # 명시적으로 숫자형(int, float) 데이터만 처리하도록 필터링 강화
        if col_type.name in ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']:
            c_min = df[col].min()
            c_max = df[col].max()
            
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float32)
    return df

# ==========================================
# 1. Train 전처리 및 군집화
# ==========================================
def preprocess_for_tree(train_path, layout_path):
    print("\n1. Train 데이터 로드 및 전처리...")
    train = pd.read_csv(train_path)
    layout = pd.read_csv(layout_path)
    
    train = pd.merge(train, layout, on='layout_id', how='left')
    train.drop('ID', axis=1, inplace=True)

    target_col = 'avg_delay_minutes_next_30m'
    y_train = train[target_col].astype('float32')
    X_train = train.drop(target_col, axis=1)
    
    for col in X_train.select_dtypes(include=['float64', 'int64']).columns:
        X_train[col].fillna(X_train[col].median(), inplace=True)

    if 'shift_hour' in X_train.columns:
        X_train['shift_hour_sin'] = np.sin(2 * np.pi * X_train['shift_hour'] / 24)
        X_train['shift_hour_cos'] = np.cos(2 * np.pi * X_train['shift_hour'] / 24)
    if 'day_of_week' in X_train.columns:
        X_train['day_of_week_sin'] = np.sin(2 * np.pi * X_train['day_of_week'] / 7)
        X_train['day_of_week_cos'] = np.cos(2 * np.pi * X_train['day_of_week'] / 7)
    X_train.drop(['shift_hour', 'day_of_week'], axis=1, inplace=True, errors='ignore') 
    
    if 'robot_utilization' in X_train.columns and 'congestion_score' in X_train.columns:
        X_train['robot_util_x_congestion'] = X_train['robot_utilization'] * X_train['congestion_score']
    if 'order_inflow_15m' in X_train.columns and 'robot_active' in X_train.columns:
        X_train['order_pressure'] = X_train['order_inflow_15m'] / (X_train['robot_active'] + 1)
    if 'low_battery_ratio' in X_train.columns and 'charge_queue_length' in X_train.columns:
        X_train['battery_stress'] = X_train['low_battery_ratio'] * X_train['charge_queue_length']
    if 'congestion_score' in X_train.columns and 'blocked_path_15m' in X_train.columns:
        X_train['congestion_impact'] = X_train['congestion_score'] * X_train['blocked_path_15m']
        
    X_train = pd.get_dummies(X_train, columns=['layout_type'], drop_first=False)
    
    X_train = reduce_mem_usage(X_train)

    print("2. 3분할(쾌적/혼잡/마비) K-Means 군집화...")
    cluster_features = [f for f in ['robot_utilization', 'congestion_score', 'charge_queue_length', 'task_reassign_15m'] if f in X_train.columns]
    
    X_cluster = X_train[cluster_features].copy()
    X_cluster.replace([np.inf, -np.inf], np.nan, inplace=True)
    X_cluster.fillna(0, inplace=True)
    
    scaler_cluster = RobustScaler() 
    X_scaled_cluster = scaler_cluster.fit_transform(X_cluster)
    
    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    X_train['cluster_id'] = kmeans.fit_predict(X_scaled_cluster).astype(np.int8)
    
    X_train.replace([np.inf, -np.inf], np.nan, inplace=True)
    X_train.fillna(0, inplace=True)
    
    del X_cluster, X_scaled_cluster
    gc.collect()
    
    return X_train, y_train

# ==========================================
# 2. 군집별 Optuna 튜닝 및 OOF 평가 (수정됨)
# ==========================================
def optimize_and_evaluate(X_train, y_train):
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    best_params_per_cluster = {}
    
    print("\n3. 군집별 Optuna 튜닝 시작...")
    
    for cid in range(3):
        mask = X_train['cluster_id'] == cid
        if not mask.any(): continue
            
        x_c = X_train[mask].drop('cluster_id', axis=1)
        y_c = y_train[mask]
        
        def objective(trial):
            params = {
                'objective': 'mae',
                'metric': 'mae',
                'boosting_type': 'gbdt',
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
                'max_depth': trial.suggest_int('max_depth', 4, 12),
                'num_leaves': trial.suggest_int('num_leaves', 15, 127),
                'subsample': trial.suggest_float('subsample', 0.7, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
                'n_estimators': 300, 
                'n_jobs': 2, 
                'verbose': -1,
                'random_state': 42
            }
            
            cv_mae = []
            for tr_idx, val_idx in KFold(n_splits=3, shuffle=True, random_state=42).split(x_c):
                # [수정] dtype 변환 버그를 막기 위해 .copy() 부활
                x_tr, y_tr = x_c.iloc[tr_idx].copy(), y_c.iloc[tr_idx].copy()
                x_val, y_val = x_c.iloc[val_idx].copy(), y_c.iloc[val_idx].copy()
                
                overall_mean = y_tr.mean()
                
                # Target Encoding & 명시적 float 형변환
                for col in ['scenario_id', 'layout_id']:
                    if col in x_tr.columns:
                        group_stats = y_tr.groupby(x_tr[col]).agg(mean=('mean'), count=('count'))
                        smoothed_means = (group_stats['mean'] * group_stats['count'] + overall_mean * 10) / (group_stats['count'] + 10)
                        
                        # [수정] map 적용 후 강제로 float32로 변환하여 문자열 껍데기 파괴
                        x_tr[col] = x_tr[col].map(smoothed_means).astype('float32')
                        x_val[col] = x_val[col].map(smoothed_means).fillna(overall_mean).astype('float32')
                
                model = lgb.LGBMRegressor(**params)
                model.fit(x_tr, y_tr, eval_set=[(x_val, y_val)], callbacks=[lgb.early_stopping(30, verbose=False)])
                cv_mae.append(mean_absolute_error(y_val, model.predict(x_val)))
                
                del x_tr, y_tr, x_val, y_val, model
                gc.collect()
                
            return np.mean(cv_mae)

        study = optuna.create_study(direction='minimize')
        study.optimize(objective, n_trials=10, gc_after_trial=True) 
        best_params_per_cluster[cid] = study.best_params
        best_params_per_cluster[cid]['n_estimators'] = 1000 
        best_params_per_cluster[cid]['objective'] = 'mae'
        best_params_per_cluster[cid]['n_jobs'] = 2 
        best_params_per_cluster[cid]['verbose'] = -1
        
        print(f"  > Cluster {cid} 최적 파라미터 찾기 완료 (OOF MAE: {study.best_value:.4f})")

    print("\n4. 최적 파라미터 장착 후 최종 5-Fold 검증 시작...")
    oof_predictions = np.zeros(len(X_train))
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
        # [수정] .copy() 부활
        x_tr, y_tr = X_train.iloc[train_idx].copy(), y_train.iloc[train_idx].copy()
        x_val, y_val = X_train.iloc[val_idx].copy(), y_train.iloc[val_idx].copy()
        
        overall_mean = y_tr.mean()
        
        # Target Encoding & 명시적 float 형변환
        for col in ['scenario_id', 'layout_id']:
            if col in x_tr.columns:
                group_stats = y_tr.groupby(x_tr[col]).agg(mean=('mean'), count=('count'))
                smoothed_means = (group_stats['mean'] * group_stats['count'] + overall_mean * 10) / (group_stats['count'] + 10)
                
                # [수정] 강제 변환
                x_tr[col] = x_tr[col].map(smoothed_means).astype('float32')
                x_val[col] = x_val[col].map(smoothed_means).fillna(overall_mean).astype('float32')

        fold_preds = np.zeros(len(x_val))
        for cid in range(3):
            tr_mask = x_tr['cluster_id'] == cid
            val_mask = x_val['cluster_id'] == cid
            if not val_mask.any() or not tr_mask.any(): continue
            
            model = lgb.LGBMRegressor(**best_params_per_cluster[cid])
            model.fit(
                x_tr[tr_mask].drop('cluster_id', axis=1), y_tr[tr_mask],
                eval_set=[(x_val[val_mask].drop('cluster_id', axis=1), y_val[val_mask])],
                callbacks=[lgb.early_stopping(50, verbose=False)]
            )
            fold_preds[val_mask] = model.predict(x_val[val_mask].drop('cluster_id', axis=1))
            
        oof_predictions[val_idx] = fold_preds
        print(f"  - Fold {fold+1} 완료 (MAE: {mean_absolute_error(y_val, fold_preds):.4f})")
        
        del x_tr, y_tr, x_val, y_val, model
        gc.collect()
        
    total_mae = mean_absolute_error(y_train, oof_predictions)
    print("\n" + "="*50)
    print(f"🏆 최종 OOF MAE: {total_mae:.4f}")
    print("="*50 + "\n")
    
    return best_params_per_cluster

if __name__ == "__main__":
    TRAIN_PATH = 'train.csv' 
    LAYOUT_PATH = 'layout_info.csv'
    
    X_train, y_train = preprocess_for_tree(TRAIN_PATH, LAYOUT_PATH)
    best_params = optimize_and_evaluate(X_train, y_train)

c:\Users\kyw41\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



1. Train 데이터 로드 및 전처리...
2. 3분할(쾌적/혼잡/마비) K-Means 군집화...

3. 군집별 Optuna 튜닝 시작...
  > Cluster 0 최적 파라미터 찾기 완료 (OOF MAE: 5.0545)
  > Cluster 1 최적 파라미터 찾기 완료 (OOF MAE: 4.3407)
  > Cluster 2 최적 파라미터 찾기 완료 (OOF MAE: 4.5966)

4. 최적 파라미터 장착 후 최종 5-Fold 검증 시작...
  - Fold 1 완료 (MAE: 4.7339)
  - Fold 2 완료 (MAE: 4.8591)
  - Fold 3 완료 (MAE: 4.7254)
  - Fold 4 완료 (MAE: 4.7633)
  - Fold 5 완료 (MAE: 4.7429)

🏆 최종 OOF MAE: 4.7649

